In [ ]:
from google.colab import files

uploaded = files.upload()

Saving test_fe.csv to test_fe.csv
Saving train_fe.csv to train_fe.csv


In [ ]:
!pip install -q xgboost

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

from xgboost import XGBClassifier

In [ ]:
train = pd.read_csv("train_fe.csv")
test = pd.read_csv("test_fe.csv")

print(train.shape)
print(test.shape)

(577347, 19)
(247435, 18)


In [ ]:
for df in [train, test]:

    df["alpha_rad"] = np.radians(df["alpha"])
    df["delta_rad"] = np.radians(df["delta"])

    df["x_coord"] = (
        np.cos(df["delta_rad"])
        * np.cos(df["alpha_rad"])
    )

    df["y_coord"] = (
        np.cos(df["delta_rad"])
        * np.sin(df["alpha_rad"])
    )

    df["z_coord"] = np.sin(
        df["delta_rad"]
    )

    df["u_div_g"] = (
        df["u"] /
        (df["g"] + 1e-6)
    )

    df["g_div_r"] = (
        df["g"] /
        (df["r"] + 1e-6)
    )

    df["r_div_i"] = (
        df["r"] /
        (df["i"] + 1e-6)
    )

    df["i_div_z"] = (
        df["i"] /
        (df["z"] + 1e-6)
    )

    df["redshift_sq"] = (
        df["redshift"] ** 2
    )

    df["redshift_cube"] = (
        df["redshift"] ** 3
    )

    df["sin_alpha"] = np.sin(
        np.radians(df["alpha"])
    )

    df["cos_alpha"] = np.cos(
        np.radians(df["alpha"])
    )

    df["sin_delta"] = np.sin(
        np.radians(df["delta"])
    )

    df["cos_delta"] = np.cos(
        np.radians(df["delta"])
    )

    df["redshift_griz"] = (
        df["redshift"]
        * df["griz"]
    )

    df["redshift_ugri"] = (
        df["redshift"]
        * df["ugri"]
    )

    df["redshift_u_g"] = (
        df["redshift"]
        * df["u_g"]
    )

    df["redshift_g_r"] = (
        df["redshift"]
        * df["g_r"]
    )

    df["redshift_r_i"] = (
        df["redshift"]
        * df["r_i"]
    )

    df["redshift_sq_griz"] = (
        df["redshift_sq"]
        * df["griz"]
    )

    df["alpha_delta"] = (
        df["alpha"]
        * df["delta"]
    )

    df["alpha_redshift"] = (
        df["alpha"]
        * df["redshift"]
    )

    df["delta_redshift"] = (
        df["delta"]
        * df["redshift"]
    )

In [ ]:
for col in [
    "spectral_type",
    "galaxy_population"
]:

    le = LabelEncoder()

    combined = pd.concat([
        train[col],
        test[col]
    ])

    le.fit(combined)

    train[col] = le.transform(
        train[col]
    )

    test[col] = le.transform(
        test[col]
    )

In [ ]:
TARGET = "class"

X = train.drop(
    columns=["id", TARGET]
)

X_test = test.drop(
    columns=["id"]
)

target_encoder = LabelEncoder()

y = target_encoder.fit_transform(
    train[TARGET]
)

In [ ]:
model = XGBClassifier(

    objective="multi:softprob",

    num_class=3,

    n_estimators=2500,

    learning_rate=0.02,

    max_depth=10,

    subsample=0.8,

    colsample_bytree=0.8,

    min_child_weight=3,

    reg_alpha=0.1,

    reg_lambda=1.0,

    random_state=42,

    tree_method="hist",

    device="cuda",

    eval_metric="mlogloss"
)

model.fit(
    X,
    y
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.02, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=10, max_leaves=None,
              min_child_weight=3, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=2500, n_jobs=None, num_class=3, ...)

In [ ]:
test_probs = model.predict_proba(
    X_test
)

max_probs = np.max(
    test_probs,
    axis=1
)

pseudo_preds = np.argmax(
    test_probs,
    axis=1
)

In [ ]:
THRESHOLD = 0.9995

mask = (
    max_probs > THRESHOLD
)

print(
    "Pseudo rows:",
    mask.sum()
)

Pseudo rows: 123564


In [ ]:
pseudo_X = X_test.loc[
    mask
].copy()

pseudo_y = pseudo_preds[
    mask
]

print(
    pseudo_X.shape
)

(123564, 41)


In [ ]:
max_probs = np.max(
    test_probs,
    axis=1
)

In [ ]:
import pandas as pd

pd.Series(max_probs).describe(
    percentiles=[
        0.90,
        0.95,
        0.97,
        0.98,
        0.99,
        0.995,
        0.999
    ]
)

,0
count,247435.000000
mean,0.976975
std,0.072264
min,0.358565
50%,0.999497
90%,0.999986
95%,0.999993
97%,0.999996
98%,0.999997
99%,0.999998


In [ ]:
X_aug = pd.concat(
    [
        X,
        pseudo_X
    ],
    axis=0
)

y_aug = np.concatenate(
    [
        y,
        pseudo_y
    ]
)

print(
    X_aug.shape
)

(700911, 41)


In [ ]:
final_model = XGBClassifier(

    objective="multi:softprob",

    num_class=3,

    n_estimators=3000,

    learning_rate=0.02,

    max_depth=10,

    subsample=0.8,

    colsample_bytree=0.8,

    min_child_weight=3,

    reg_alpha=0.1,

    reg_lambda=1.0,

    random_state=42,

    tree_method="hist",

    device="cuda",

    eval_metric="mlogloss"
)

final_model.fit(
    X_aug,
    y_aug
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.02, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=10, max_leaves=None,
              min_child_weight=3, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=3000, n_jobs=None, num_class=3, ...)

In [ ]:
final_probs = final_model.predict_proba(
    X_test
)

final_preds = np.argmax(
    final_probs,
    axis=1
)

In [ ]:
import pandas as pd

pseudo_classes = pd.Series(
    target_encoder.inverse_transform(
        pseudo_preds
    )
)

pseudo_classes.value_counts(normalize=True)

,proportion
GALAXY,0.655271
QSO,0.202211
STAR,0.142518


In [ ]:
train["class"].value_counts(normalize=True)

,proportion
class,
GALAXY,0.653818
QSO,0.202899
STAR,0.143283


In [ ]:
submission = pd.DataFrame({

    "id":
        test["id"],

    "class":
        target_encoder.inverse_transform(
            final_preds
        )
})

submission.to_csv(
    "submission_xgb_pseudo.csv",
    index=False
)

submission.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


In [ ]:
prob_df = pd.DataFrame(
    final_probs,
    columns=target_encoder.classes_
)

prob_df.insert(
    0,
    "id",
    test["id"]
)

prob_df.to_csv(
    "xgb_pseudo_prob.csv",
    index=False
)

In [ ]:
from google.colab import files

files.download(
    "submission_xgb_pseudo.csv"
)

files.download(
    "xgb_pseudo_prob.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>